<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB18_Case_Study_ECMWF_Predicting_Wave_Height_from_Wind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB18 · Class 18 — Case Study: ECMWF Weather Data, Predicting Wave Height from Wind**

## Block 4: Proyectos — Case Studies (opening)

Blocks 2–3 taught the toolkit; Block 4 applies it to complete real case studies — closer to how you'll actually use this material after the course. This first case study uses **real ECMWF reanalysis data**, retrieved live from the same Copernicus Climate Data Store professionals use operationally.

**The real-world problem**: ocean waves are generated by wind — how big they grow depends on wind speed, how long it has blown (duration), and how much open water it has crossed (fetch). If a data-driven model can learn that real physical relationship from real data, it becomes genuinely useful: `a fast, cheap way to estimate sea state from wind alone`, valuable whenever running a full physics-based wave model (like WAM or WaveWatch III) isn't practical — a quick onboard estimate with limited connectivity, filling a gap where wave-model output isn't available yet, or sanity-checking a more expensive model's output.

This class has **two parts**, and the difference between them matters:
- **Part A** (Sections 3–10) validates a modeling approach using real ERA5 data for **one calendar date — January 15 — pooled across five real years (2019–2023)**, where we already know the true wave height everywhere. Using several years of the *same* date, rather than a single snapshot, matters: wave height depends heavily on the time of year, so pooling multiple years of the same date teaches the model the real range of winter wind-wave conditions at this location, instead of memorizing one single day's weather. This is the same "known-label" setup as `NB07`'s fuel-consumption model or `NB10`'s hull-resistance model, used to build and sanity-check a method *before* trusting it on something genuinely unknown.
- **Part B** (Section 11) is the actual test: applying that validated approach to **January 15, 2024 — the same calendar date, but a year the model has never seen** — predicting wave height from wind **alone**, and only revealing the true values afterward to check. Holding out a *year*, not a different season, is what makes this a fair test: it checks whether the model generalizes to genuinely new weather (a year it never saw), without also asking it to generalize across a season it was never trained on.

You will choose the modeling approach yourself in Part A, using `NB17`'s decision framework — this class does not tell you which architecture to use.

> **Important — do this before class**: register for a free Copernicus CDS account and generate your API key at [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) (an institutional email is recommended). Registration/approval can take a little time, so do this **before** the session, not during it.

### Learning objectives

By the end of this class, students will be able to:
- Explain, physically, why wind can predict wave height, and why that relationship has real operational value.
- Explain why a fair generalization test must hold out a *year*, not a different season, when the target variable is seasonal.
- Retrieve real, multi-year reanalysis data from the Copernicus Climate Data Store via the `cdsapi`.
- Reshape gridded NetCDF climate data, pooled across several years, into a flat, ML-ready table.
- Apply `NB17`'s decision framework to a new problem and justify a modeling choice.
- Distinguish a methodology-validation exercise (known labels, training years) from a genuine held-out-year forecast, and explain why a real project needs both.
- Apply a trained model to forecast the same calendar date in a year it has never seen — without looking at its true values until after predicting — and evaluate it honestly.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, Block 4 introduction, today's roadmap | 5 min | Theory |
| 2 | What is ECMWF/ERA5, and why "wind → wave height" is a real problem | 10 min | Theory |
| 3 | Setting up Copernicus CDS API access | 10 min | Practice |
| 4 | Downloading real, multi-year ERA5 data | 5 min | Practice |
| 5 | Exploring the NetCDF grid | 5 min | Practice |
| 6 | Reshaping and pooling five years into a flat table | 10 min | Practice |
| 7 | Exploring the real, flattened dataset | 15 min | Practice |
| 8 | Applying `NB17`'s decision framework | 5 min | Theory + Practice |
| 9 | Hands-on: training and comparing models (methodology validation) | 15 min | Practice |
| 10 | Evaluation and a spatial sanity check (methodology validation) | 15 min | Practice |
| 11 | A genuinely predictive test: forecasting a held-out year | 20 min | Practice |
| 12 | Summary, homework, what's next in Block 4 | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap and Block 4 introduction

- **Block 2** (`NB02`–`NB10`): the classical ML toolkit.
- **Block 3** (`NB11`–`NB17`): Deep Learning, closing with a decision framework for choosing between everything learned so far.
- **Block 4** (starting today): complete case studies, applying that toolkit to new real problems.

Block 4 is also where the course's evaluation catches up with the teaching: your **individual final project** (40% of the grade — an unseen case study you present and defend) uses a *different* real dataset from the ones taught in class, and the **case-study submission** (20%) lets you pick and resubmit your own resolution of *any* notebook worked in class, including this one. From here on, some class sessions will be new case studies like this one, and others will be supervised time for you to work on your own project, with a short checkpoint due every session — ask your instructor for the exact schedule.

---

## 2. What is ECMWF/ERA5, and why "wind → wave height" is a real problem

The **[European Centre for Medium-Range Weather Forecasts (ECMWF)](https://en.wikipedia.org/wiki/ECMWF)** is an intergovernmental organization and one of the world's leading centers for numerical weather prediction. Its **[ERA5](https://en.wikipedia.org/wiki/ERA5)** dataset is a *reanalysis*: not a raw observation and not a forecast, but a physically consistent reconstruction of past atmospheric and ocean-surface conditions, produced by combining millions of real historical observations with a numerical weather model. In practice, this means ERA5 gives us **realistic, physically consistent global weather and sea-state data for any place and time since 1940** — exactly the kind of real environmental data a naval or ocean engineer would use for route planning, structural load estimates, or historical weather analysis.

### Why "wind → wave height" specifically

`Waves form because wind transfers energy to the sea surface`. How large they grow depends on three real physical factors: wind **speed**, how long it has been blowing (**duration**), and how much open water it has crossed (**fetch**). This is established physical oceanography — not a coincidence chosen to make a tidy classroom problem.

A model that learns this relationship well from real data has genuine operational value. Full physics-based wave models (like WAM or WaveWatch III) are the accurate, trusted standard, but they are computationally expensive and need dedicated infrastructure to run — they are not something you casually re-run for a quick estimate. A lightweight, data-driven wind→wave model can never replace them for serious forecasting, but it is useful anywhere a fast, rough estimate beats no estimate at all: a quick onboard check with limited connectivity, filling a gap where wave-model output isn't available for a given time or place, or sanity-checking a more expensive model's output before trusting it.

### One methodological detail worth spelling out before any code

Wave height depends strongly on the time of year — a January storm and a July calm day are not the same physical regime, and a model trained on one season has no real basis for predicting a different one; testing that would only tell us the obvious (seasons differ), not whether the model actually generalizes. To test genuine year-to-year generalization *without* also crossing seasons, this class fixes the calendar date — January 15 — and varies only the **year**: training on several past years of that date, then forecasting the same date in a year never seen. That isolates the question we actually care about: does the model generalize to new *weather* (a year it hasn't seen), holding the *season* constant?

That is the actual question this class answers — not "can we fit a curve to some numbers," but "does a real, physically-motivated relationship show up clearly enough in real data for a model to learn it, and can that model then say something true about a year it has never seen?" Part A below builds and validates the approach, pooling several real winters; Part B (Section 11) is where we actually answer it, on a genuinely held-out year.

---

## 3. Setting up Copernicus CDS API access

With your free API key from [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) ready, install the client and save your credentials for this session:

In [ ]:
%pip install -q cdsapi xarray netCDF4 cartopy

Run the cell below and paste your key **when prompted** — it uses `getpass`, which hides what you type/paste and, more importantly, means your real key is **never written into this notebook's code or saved output**. Unlike a hardcoded key sitting in a cell (which is exactly how this repository leaked a real API key earlier in this course's history — see the very first security fixes in this project), a `getpass` prompt only exists in the running session's memory; nothing about it ends up in this file when you save it, share it, or push it to GitHub.

1. Go to [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) and log in.
2. Copy your **Personal Access Token** (a long string of letters, numbers, and dashes).
3. Run the cell below. A hidden input box appears — paste your token there and press Enter.

In [ ]:
import os
from getpass import getpass

CDS_API_KEY = getpass("Paste your CDS API key from https://cds.climate.copernicus.eu/profile (input hidden): ").strip()

if not CDS_API_KEY:
    raise ValueError(
        "No key entered. Get one from https://cds.climate.copernicus.eu/profile "
        "and re-run this cell."
    )

cdsapirc = f"url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n"

with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(cdsapirc)

print("CDS API key saved for this session (not stored anywhere in this notebook).")

---

## 4. Downloading real, multi-year ERA5 data

Request 10 m wind components and significant wave height, over the North Atlantic/Western European shelf — a real, naval-relevant region (English Channel, Bay of Biscay, North Sea) — for **January 15, 12:00 UTC, across five real years (2019–2023)**. The CDS API accepts a list of years in a single request, so this is still one download, just one that spans several real winters instead of a single day:

In [ ]:
import cdsapi

client = cdsapi.Client()

TRAINING_YEARS = ["2019", "2020", "2021", "2022", "2023"]

client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "significant_height_of_combined_wind_waves_and_swell",
        ],
        "year": TRAINING_YEARS,
        "month": "01",
        "day": "15",
        "time": ["12:00"],
        "area": [60, -20, 35, 10],  # North, West, South, East
    },
    "era5_training_years.nc",
)

The Climate Data Store sometimes returns a single NetCDF file and sometimes a zip archive splitting variables by internal data "stream" — handle both cases so the rest of the notebook doesn't depend on which one you got:

In [ ]:
import zipfile
import glob

target = "era5_training_years.nc"
extract_dir = "."

if zipfile.is_zipfile(target):
    extract_dir = "era5_extracted"
    with zipfile.ZipFile(target) as zf:
        zf.extractall(extract_dir)
    print("Zip archive detected and extracted:", os.listdir(extract_dir))
else:
    print("Single NetCDF file, no extraction needed.")

nc_files = glob.glob(os.path.join(extract_dir, "*.nc")) or [target]
print("NetCDF files:", nc_files)

---

## 5. Exploring the NetCDF grid

Open every file found and identify which one holds the wave variable (`swh`) and which holds the wind components (`u10`/`v10`) — this also protects the notebook against the file(s) coming back in a different arrangement than expected:

In [ ]:
import xarray as xr

datasets = [xr.open_dataset(f) for f in nc_files]
for i, ds in enumerate(datasets):
    print(f"File {i}: variables = {list(ds.data_vars)}, dims = {dict(ds.sizes)}")

wave_ds = next(ds for ds in datasets if "swh" in ds.data_vars)
wind_ds = next(ds for ds in datasets if "u10" in ds.data_vars and "v10" in ds.data_vars)

Plot the raw wave-height grid for one example training year — the same first sanity check any real gridded dataset deserves before modeling anything. The other four years are pooled in silently for now and get their turn in Part 6:

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

example_year_idx = 0  # first of the 5 training years
swh_example = wave_ds["swh"].isel(valid_time=example_year_idx)
print("Showing:", str(wave_ds.valid_time.isel(valid_time=example_year_idx).values)[:10])

lon_min, lon_max = float(swh_example.longitude.min()), float(swh_example.longitude.max())
lat_min, lat_max = float(swh_example.latitude.min()), float(swh_example.latitude.max())

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

im = ax.imshow(swh_example.values, origin="upper", extent=[lon_min, lon_max, lat_min, lat_max],
                cmap="viridis", transform=ccrs.PlateCarree())

ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.gridlines(draw_labels=True)

plt.colorbar(im, ax=ax, orientation="vertical", pad=0.05, shrink=0.7, label="Significant wave height (m)")
plt.title("Real ERA5 significant wave height -- one example training year")
plt.tight_layout()
plt.show()

With real coastlines and country borders drawn in, the blank/NaN region visibly lines up with land (France, the UK, Spain, Portugal) — wave height is only defined over open water, which becomes relevant in a moment. Being able to see *where* on Earth this data actually is `matters for a naval/ocean case study, not just as decoration`: it's how you'd recognize a fetch-limited enclosed sea (the English Channel, the North Sea) versus open Atlantic swell later in Part 7.

---

## 6. Reshaping and pooling five years into a flat table

Every model we've used since `NB02` expects a flat table: one row per example, one column per feature. A spatial grid needs reshaping first — every `(latitude, longitude)` cell becomes one row, and since we now have **5 training years**, each year's grid gets reshaped and stacked into one combined table. Deliberately, no `year` column is kept as a feature: we want the model to learn the general wind→wave relationship for this time of year, not to memorize a per-year offset.

One real wrinkle: ERA5's wave parameters (like `swh`) and its surface/atmospheric parameters (like `u10`/`v10`) are not always delivered on identically-shaped grids, even when requested together over the same area — a known quirk of how ECMWF's forecasting system represents ocean waves internally. Check both datasets' grid sizes first:

In [ ]:
print("wave_ds grid:", wave_ds.sizes)
print("wind_ds grid:", wind_ds.sizes)

If the two sizes above differ, the wind field needs interpolating onto the wave field's grid before they can share one table — done automatically below regardless of whether they matched or not, so this notebook works either way:

In [ ]:
import numpy as np
import pandas as pd

# Align the wind field onto the wave field's exact grid, whether or not they
# originally matched -- interpolation is a no-op if the grids already agree.
wind_ds_aligned = wind_ds.interp(latitude=wave_ds.latitude, longitude=wave_ds.longitude)
lat_grid, lon_grid = np.meshgrid(wave_ds.latitude.values, wave_ds.longitude.values, indexing="ij")

n_years = wave_ds.sizes["valid_time"]
print(f"Pooling {n_years} training years into one table...")

year_frames = []
for t in range(n_years):
    u10 = wind_ds_aligned["u10"].isel(valid_time=t).values
    v10 = wind_ds_aligned["v10"].isel(valid_time=t).values
    swh_vals = wave_ds["swh"].isel(valid_time=t).values

    assert u10.shape == swh_vals.shape == lat_grid.shape, (
        f"Grid shapes still don't match for year index {t}: u10 {u10.shape}, "
        f"swh {swh_vals.shape}, lat_grid {lat_grid.shape} -- inspect wave_ds/wind_ds.sizes above."
    )

    wind_speed = np.sqrt(u10 ** 2 + v10 ** 2)
    wind_direction = (np.degrees(np.arctan2(u10, v10)) + 360) % 360

    year_frames.append(pd.DataFrame({
        "latitude": lat_grid.ravel(),
        "longitude": lon_grid.ravel(),
        "u10": u10.ravel(),
        "v10": v10.ravel(),
        "wind_speed": wind_speed.ravel(),
        "wind_direction": wind_direction.ravel(),
        "swh": swh_vals.ravel(),
    }))

grid_df = pd.concat(year_frames, ignore_index=True).dropna()
print(grid_df.shape)
grid_df.head()

`dropna()` removed every land grid cell across all 5 years in one step — `a real, physically meaningful reason for missing data`, different in kind from `NB13`'s sensor-file mismatch but handled the same way: understand *why* it's missing before deciding what to do about it. `grid_df` now holds roughly 5× the ocean cells a single snapshot would give — the same location's wind/wave relationship, sampled across 5 real winters instead of one.

---

## 7. Exploring the real, flattened dataset

`NB02`'s three starting questions, once more, on this real, multi-year pooled dataset:

In [ ]:
grid_df.describe()

And the relationship the whole class hinges on — does wave height actually track wind speed in this real data?

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(grid_df["wind_speed"], grid_df["swh"], alpha=0.3, s=10)
plt.xlabel("Wind speed (m/s)")
plt.ylabel("Significant wave height (m)")
plt.title("Wind speed vs. wave height, every real grid cell")
plt.show()

**Try it yourself**: put a number on what the scatter plot shows — compute the actual correlation between `wind_speed` and `swh` across every real pooled grid cell.

In [ ]:
correlation = grid_df[["wind_speed", "swh"]].corr().iloc[0, 1]
print(f"Correlation between wind speed and wave height: {correlation:.3f}")


**Read your own plot**: is the relationship a clean line, a noisy trend, or close to no relationship at all? Real fetch-limited seas (an enclosed sea like parts of the English Channel, where wind doesn't have room to build up large waves) can look quite different from open Atlantic swell in the same snapshot — `latitude`/`longitude` may end up mattering as much as wind speed itself.

---

## 8. Applying `NB17`'s decision framework

Before writing any model code, run this real problem through `NB17`'s three questions:

1. **Labels?** Yes — `swh` is a real, known target for every grid cell, in every training year.
2. **Data shape?** Tabular — each row is a single grid cell's features, not an image or a sequence (even though it originated from a spatial grid, we've already flattened it).
3. **Data volume?** Several thousand ocean grid cells after `dropna()`, now pooled across 5 years — moderate, not huge, but larger than any single snapshot would give us.

`NB17`'s framework, applied honestly, points toward **classical ML** as at least a strong baseline here — plausibly the right final answer too, exactly as it was for `NB10`'s yacht data in `NB17`'s own experiment. A neural network remains a legitimate choice to *also* try and compare, but the framework gives no reason to assume it will automatically win. **Your task**: pick at least one classical model and justify your choice out loud (to a classmate or your instructor) before writing the training code below.

---

## 9. Hands-on: training and comparing models

A leakage check first — `wind_speed` and `wind_direction` were both *derived* from `u10`/`v10`, so using all four together would just be feeding the model the same information twice in different forms, the same redundancy principle from `NB07`'s `CO2_emissions` example. Use `u10`/`v10` **or** `wind_speed`/`wind_direction`, not both representations at once:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = ["latitude", "longitude", "wind_speed", "wind_direction"]
X = grid_df[feature_cols]
y = grid_df["swh"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

> **A real limitation worth naming, not hiding**: this is a *random* split of both spatially correlated data (nearby grid cells have similar wave heights) and, now, repeated locations across years (the same `(latitude, longitude)` cell appears once per training year, and its 5 versions aren't fully independent of each other either). `A stricter evaluation would hold out an entire spatial region, or an entire year, instead of random rows` (try both as homework). We proceed with the random split for today's methodology check, honestly labeled as a simplification — Part 11's held-out-year test is exactly the stricter version of this, done properly.

Compare a linear baseline against a tree ensemble — the classical candidates `NB17`'s framework favored:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in candidate_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring="r2")
    cv_results[name] = scores

pd.DataFrame(cv_results).mean().sort_values(ascending=False)

**Try it yourself**: add a Gradient Boosting Regressor (`NB10`'s third candidate) to the comparison above — does it beat both Linear Regression and Random Forest here?

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

candidate_models["Gradient Boosting"] = GradientBoostingRegressor(random_state=42)
gb_scores = cross_val_score(candidate_models["Gradient Boosting"], X_train_scaled, y_train, cv=cv, scoring="r2")
cv_results["Gradient Boosting"] = gb_scores

pd.DataFrame(cv_results).mean().sort_values(ascending=False)


---

## 10. Evaluation and a spatial sanity check

This still evaluates on rows drawn from the *same* pool of training years (2019–2023) — a fair check of the modeling approach, but not yet a genuine test of forecasting a new year (that's Section 11, right after this). Fit the stronger candidate on the full training set and evaluate once on the untouched test set:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = RandomForestRegressor(n_estimators=200, random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

print(f"MAE:  {mean_absolute_error(y_test, y_pred):.3f} m")
print(f"RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.3f} m")
print(f"R2:   {r2_score(y_test, y_pred):.3f}")

**Try it yourself**: which feature does the Random Forest rely on most — latitude, longitude, wind speed, or wind direction?

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances


A spatial sanity check — plot the test-set residuals back on the map. Errors scattered randomly suggest a reasonably unbiased model; errors clustered in one region (say, the English Channel specifically) would suggest the model is systematically missing something about that area's sea state:

In [ ]:
residuals = y_test.values - y_pred

fig = plt.figure(figsize=(9, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

sc = ax.scatter(X_test["longitude"], X_test["latitude"], c=residuals, cmap="coolwarm",
                 vmin=-abs(residuals).max(), vmax=abs(residuals).max(), s=15,
                 transform=ccrs.PlateCarree())

ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.gridlines(draw_labels=True)

plt.colorbar(sc, ax=ax, orientation="vertical", pad=0.05, shrink=0.7, label="Residual (actual - predicted), m")
plt.title("Where does the model's error concentrate?")
plt.tight_layout()
plt.show()

---

## 11. A genuinely predictive test: forecasting a held-out year

Sections 5–10 pooled five real winters (2019–2023) of the *same calendar date* — January 15 — to validate a modeling approach. That's a fair methodology check, and pooling years (rather than a single snapshot) already gives the model exposure to real year-to-year variability in winter wind-wave conditions. But every one of those years was available during training; we haven't yet tested the model on a year it genuinely never saw.

Now we do the real thing: fetch **January 15, 2024** — the same calendar date, deliberately, so we test generalization across *years*, not across *seasons* — predict its wave heights from wind **alone**, and only reveal the true values afterward to check. This mirrors how a model would actually be used operationally: trained once on historical winters, then applied to a new winter whose outcome isn't known yet.

In [ ]:
client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "significant_height_of_combined_wind_waves_and_swell",
        ],
        "year": "2024",
        "month": "01",
        "day": "15",
        "time": ["12:00"],
        "area": [60, -20, 35, 10],
    },
    "era5_forecast_target.nc",
)

This file technically contains the real wave heights for `2024-01-15` too — but to keep this a genuine test, **we will not open the `swh` variable until after making our prediction**. From here until the reveal step, treat this as if only wind data were available:

In [ ]:
target2 = "era5_forecast_target.nc"
extract_dir2 = "."

if zipfile.is_zipfile(target2):
    extract_dir2 = "era5_forecast_extracted"
    with zipfile.ZipFile(target2) as zf:
        zf.extractall(extract_dir2)

nc_files2 = glob.glob(os.path.join(extract_dir2, "*.nc")) or [target2]

datasets2 = [xr.open_dataset(f) for f in nc_files2]
wind_ds2 = next(ds for ds in datasets2 if "u10" in ds.data_vars and "v10" in ds.data_vars)
wave_ds2 = next(ds for ds in datasets2 if "swh" in ds.data_vars)  # not opened/used yet

print("New date's wind grid:", wind_ds2.sizes)

Build the feature table exactly as in Part 6 — but we need a way to know which grid cells are ocean (versus land, where no wave height exists) *without* looking at this new year's `swh`. Land and sea don't move from one year to the next, so reuse the ocean-cell locations already established from the pooled training years in Part 6, purely from geography:

In [ ]:
wind_ds2_aligned = wind_ds2.interp(latitude=wave_ds2.latitude, longitude=wave_ds2.longitude)
u10_2 = wind_ds2_aligned["u10"].isel(valid_time=0).values
v10_2 = wind_ds2_aligned["v10"].isel(valid_time=0).values
lat_grid2, lon_grid2 = np.meshgrid(wave_ds2.latitude.values, wave_ds2.longitude.values, indexing="ij")

wind_speed2 = np.sqrt(u10_2 ** 2 + v10_2 ** 2)
wind_direction2 = (np.degrees(np.arctan2(u10_2, v10_2)) + 360) % 360

target_df = pd.DataFrame({
    "latitude": lat_grid2.ravel(),
    "longitude": lon_grid2.ravel(),
    "wind_speed": wind_speed2.ravel(),
    "wind_direction": wind_direction2.ravel(),
})

# Ocean/land is geography, not weather -- reuse the cells already known to be
# ocean from Part 6's training snapshot, without looking at this date's swh.
ocean_cells = grid_df[["latitude", "longitude"]].drop_duplicates()
target_df = target_df.merge(ocean_cells, on=["latitude", "longitude"], how="inner")
print(target_df.shape)

Predict, using the model already trained in Part 9 — no retraining, no peeking:

In [ ]:
X_target = target_df[feature_cols]
X_target_scaled = scaler.transform(X_target)
target_df["predicted_swh"] = best_model.predict(X_target_scaled)
target_df.head()

**Now, and only now**, reveal the real wave heights for this date, to see how well the prediction actually did:

In [ ]:
swh2 = wave_ds2["swh"].isel(valid_time=0)
lat_grid2b, lon_grid2b = np.meshgrid(wave_ds2.latitude.values, wave_ds2.longitude.values, indexing="ij")
truth_df = pd.DataFrame({
    "latitude": lat_grid2b.ravel(),
    "longitude": lon_grid2b.ravel(),
    "true_swh": swh2.values.ravel(),
})

comparison = target_df.merge(truth_df, on=["latitude", "longitude"], how="left")

print(f"MAE:  {mean_absolute_error(comparison['true_swh'], comparison['predicted_swh']):.3f} m")
print(f"RMSE: {mean_squared_error(comparison['true_swh'], comparison['predicted_swh']) ** 0.5:.3f} m")
print(f"R2:   {r2_score(comparison['true_swh'], comparison['predicted_swh']):.3f}")

**Try it yourself**: MAE and RMSE are in meters — put the held-out-year error in relative terms too, as a percentage of the true mean wave height, for a scale-independent read of how big the miss actually is.

In [ ]:
mean_true_swh = comparison["true_swh"].mean()
mae_pct = mean_absolute_error(comparison["true_swh"], comparison["predicted_swh"]) / mean_true_swh * 100

print(f"Mean true wave height: {mean_true_swh:.2f} m")
print(f"MAE as % of mean wave height: {mae_pct:.1f}%")


**Compare these numbers to Part 10's.** Are they close, or noticeably worse? Because Part 11 holds out a *year*, not a *season*, this is a fair, like-for-like comparison — any drop here reflects genuine year-to-year variability in winter conditions (a stormier or calmer winter than 2019–2023 typically saw), not the model being asked to generalize across a completely different climate regime it never saw. That makes this result directly trustworthy for the question a naval engineer would actually ask: "if I trained this on past winters, how much would I trust it on a new one?" A visual side-by-side makes the comparison concrete:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), subplot_kw={"projection": ccrs.PlateCarree()})

for ax, col, title in zip(axes, ["true_swh", "predicted_swh"], ["True (revealed)", "Predicted"]):
    sc = ax.scatter(comparison["longitude"], comparison["latitude"], c=comparison[col],
                     cmap="viridis", vmin=comparison["true_swh"].min(), vmax=comparison["true_swh"].max(),
                     s=15, transform=ccrs.PlateCarree())
    ax.coastlines(resolution="110m")
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.set_title(f"{title} wave height, 2024-01-15")

fig.colorbar(sc, ax=axes, orientation="horizontal", pad=0.08, shrink=0.6, label="Significant wave height (m)")
plt.show()

This is the real test this notebook set out to run: `not fitting a model to data it can already see`, but forecasting a genuinely new **year** — with season held constant — and checking only afterward. That is the honest, complete answer to "can wind alone predict wave height?", and it is a fair test precisely because we were careful about *what* we held out.

---

## 12. Summary, homework, and what's next in Block 4

- ERA5 reanalysis provides real, physically consistent historical weather/ocean data, retrieved live via the Copernicus CDS API — a genuine professional workflow, not a teaching shortcut.
- Wind predicts wave height because it physically generates waves (speed, duration, fetch) — a real relationship with real operational value as a cheap surrogate for expensive physics-based wave models.
- Wave height is seasonal, so a fair generalization test must hold out a *year*, not a different time of year — pooling several years of the same calendar date for training, then testing on a held-out year, isolates real year-to-year variability from the much larger (and unfair to test for) seasonal effect.
- Gridded NetCDF data reshapes into a flat table exactly like any other dataset once you understand its dimensions — `dropna()` removing land cells was a physically meaningful cleaning step, not an arbitrary one.
- `NB17`'s decision framework, applied to a genuinely new problem, pointed toward classical ML — and a quick cross-validated comparison backed that up.
- Evaluating on held-out rows from the *same pool of training years* (Part 10) validates a method; forecasting a genuinely new *year* (Part 11) — never seen, land/ocean determined from geography alone, true values revealed only after predicting — is the real, honest test of generalization.
- A residual map and a predicted-vs-true side-by-side map are spatial-data-specific interpretation tools, alongside the usual MAE/RMSE/R².

### What's next in Block 4

The next taught sessions in this block cover other real case studies (historical voyage records, terrain/elevation data). The sessions *between* them are your supervised project time — bring real, checkpointable progress on your individual final project every time, per the schedule in `Final_Project_Wave_Height_Forecasting_STARTER.ipynb`'s rubric (note: that project uses a **different** real dataset from today's, by design — see that notebook's rules).

## Homework / Practice Ideas

1. Change the `area` in Part 4 to a different real region (e.g., the Mediterranean, or waters near your own country) and re-run the notebook — does the wind-vs-wave relationship from Part 7 look similar?
2. Implement the stricter spatial holdout suggested in Part 9: split by a longitude threshold (e.g., train on everything west of -5°, test on everything east of it) instead of a random split — how much does the reported R² change?
3. Add a small MLP (`NB11` style) to Part 9's comparison — does it beat the Random Forest here, and does that match or contradict `NB17`'s general expectation?
4. In Part 11, try a different held-out year as the genuine forecast target (e.g., `2018-01-15`, added to or swapped with the training pool) — does forecast accuracy stay consistent across different held-out years?
5. Extend Part 4's request to a small window of days around January 15 (e.g., January 10–20) for each training year, instead of a single day — does the extra data per year noticeably change Part 10's or Part 11's results?
6. Using the residual map from Part 10 and the predicted-vs-true maps from Part 11, identify the region where the *genuine forecast* (Part 11) struggles most, and propose (in a markdown cell, no code needed) a feature that might explain that error.

> ***As always: a new case study is only "solved" once you can explain both what the model got right and where it struggled — and, per Part 11, only once you've tested it on something it never got to see coming, in a way that's actually a fair test.***
